# Notebook 1 — SNOMED CT Concept Selection

Retrieves breast cancer–related concepts from the Snowstorm REST API across multiple
semantic tag groups (disorders, mammography procedures, screening procedures, family
history situations, and standalone findings/situations), then computes pairwise
ontological distances via the IS-A hierarchy.

**Query groups**
- `<< 254837009` — Malignant neoplasm of breast (disorders)
- `<< 71651007` — Mammography (procedures)
- `<< 268547008` — Screening for malignant neoplasm of breast (procedures)
- `<< 313102001` — Family history of neoplasm of breast (situations)
- Standalone concept IDs — at-risk, detection, gene/marker, and clinical/admin findings

**Output files**
- `data/concepts.csv` — concept_id, preferred_term, fsn, semantic_tag, query_group, ancestor_count
- `data/ontological_distances.csv` — n×n pairwise distance matrix

## 1a. Configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import time

SNOWSTORM_URL = "https://snowstorm.snomed.consultologist.ai"
BRANCH = "MAIN"
LIMIT = 500

# ECL queries — each retrieves a subtree of descendants
ECL_QUERIES = {
    "disorders":      "<< 254837009",
    "mammography":    "<< 71651007",
    "screening":      "<< 268547008",
    "family_history": "<< 313102001",
}

# Standalone concept IDs not covered by the ECL subtrees above
STANDALONE_IDS = [
    # suspected (situation)
    "134405005", "366980001", "12275351000119103",
    # at risk (finding)
    "866242004", "705089007",
    # detection (finding)
    "10011000087104", "10041000087103", "10051000087100",
    # gene / marker (finding)
    "10651000087109", "412734009", "412738007",
    "412736006", "412739004", "445333001", "445180002",
    # clinical / admin (finding)
    "406100007", "9901000087105", "724451007",
]

## 1b. Retrieve concepts (disorders, mammography, screening, family history, standalones)

In [ ]:
def get_concepts_by_ecl(ecl: str, limit: int = 500) -> list[dict]:
    url = f"{SNOWSTORM_URL}/{BRANCH}/concepts"
    r = requests.get(url, params={"ecl": ecl, "limit": limit, "active": True})
    r.raise_for_status()
    return r.json()["items"]


def get_concepts_by_ids(concept_ids: list[str]) -> list[dict]:
    """Fetch a batch of concepts by ID using the conceptIds parameter."""
    url = f"{SNOWSTORM_URL}/{BRANCH}/concepts"
    # API accepts repeated conceptIds params; pass as comma-joined list
    r = requests.get(url, params={"conceptIds": concept_ids, "limit": len(concept_ids), "active": True})
    r.raise_for_status()
    return r.json()["items"]


# --- Collect all concepts, deduplicating by conceptId ---
seen_ids = set()
concepts = []
concept_groups = {}   # conceptId → group name

for name, ecl in ECL_QUERIES.items():
    batch = get_concepts_by_ecl(ecl, limit=LIMIT)
    added = [c for c in batch if c["conceptId"] not in seen_ids]
    seen_ids.update(c["conceptId"] for c in added)
    concepts.extend(added)
    for c in added:
        concept_groups[c["conceptId"]] = name
    print(f"  {name:15s}: {len(batch):3d} retrieved, {len(added):3d} new")

# Standalone IDs not already covered
missing_ids = [cid for cid in STANDALONE_IDS if cid not in seen_ids]
if missing_ids:
    standalone = get_concepts_by_ids(missing_ids)
    seen_ids.update(c["conceptId"] for c in standalone)
    concepts.extend(standalone)
    for c in standalone:
        concept_groups[c["conceptId"]] = "standalone"
    print(f"  {'standalones':15s}: {len(missing_ids):3d} requested, {len(standalone):3d} retrieved")

print(f"\nTotal unique concepts: {len(concepts)}")

In [ ]:
# Probe: check whether semanticTag is a top-level field in raw API responses
probe_id = concepts[0]["conceptId"]

for endpoint in [
    f"{SNOWSTORM_URL}/{BRANCH}/concepts/{probe_id}",
    f"{SNOWSTORM_URL}/browser/{BRANCH}/concepts/{probe_id}",
]:
    r = requests.get(endpoint)
    if r.ok:
        data = r.json()
        print(f"\n{endpoint}")
        print(f"  Top-level keys: {list(data.keys())}")
        print(f"  semanticTag present: {'semanticTag' in data}")
        if "semanticTag" in data:
            print(f"  semanticTag value: {data['semanticTag']}")
        print(f"  fsn: {data.get('fsn', {}).get('term', 'N/A')}")
    else:
        print(f"\n{endpoint} → {r.status_code}")

In [ ]:
import re

concept_ids     = [c["conceptId"] for c in concepts]
preferred_terms = [c["pt"]["term"] for c in concepts]
fsns            = [c["fsn"]["term"] for c in concepts]
semantic_tags   = [re.search(r'\(([^)]+)\)$', f).group(1) for f in fsns]
query_groups    = [concept_groups[cid] for cid in concept_ids]

df_concepts = pd.DataFrame({
    "concept_id":     concept_ids,
    "preferred_term": preferred_terms,
    "fsn":            fsns,
    "semantic_tag":   semantic_tags,
    "query_group":    query_groups,
}).sort_values("semantic_tag").reset_index(drop=True)

# Re-derive ordered lists after sort
concept_ids     = df_concepts["concept_id"].tolist()
preferred_terms = df_concepts["preferred_term"].tolist()
fsns            = df_concepts["fsn"].tolist()

print(df_concepts.head(10))
print(df_concepts["semantic_tag"].value_counts())
print(df_concepts["query_group"].value_counts())

## 1c. Compute pairwise ontological distances

Distance via shared ancestor (LCA) approximation:
```
distance(A, B) = |ancestors(A)| + |ancestors(B)| − 2 × |ancestors(A) ∩ ancestors(B)|
```

In [ ]:
# Probe: find the working ancestors endpoint for this server
test_id = concept_ids[0]
candidates = [
    f"{SNOWSTORM_URL}/{BRANCH}/concepts/{test_id}/ancestors",
    f"{SNOWSTORM_URL}/browser/{BRANCH}/concepts/{test_id}/ancestors",
    f"{SNOWSTORM_URL}/snowstorm/snomed-ct/{BRANCH}/concepts/{test_id}/ancestors",
]
for url in candidates:
    r = requests.get(url, params={"form": "inferred", "limit": 5})
    print(f"{r.status_code}  {url}")

In [ ]:
def get_ancestors(concept_id: str) -> tuple[set[str] | None, int | None]:
    url = f"{SNOWSTORM_URL}/browser/{BRANCH}/concepts/{concept_id}/ancestors"
    r = requests.get(url, params={"form": "inferred", "limit": 200})
    if r.status_code in (404, 500):
        return None, r.status_code
    r.raise_for_status()
    data = r.json()
    items = data if isinstance(data, list) else data["items"]
    return {c["conceptId"] for c in items}, 200

ancestor_sets = {}
skipped = {404: [], 500: []}
for i, cid in enumerate(concept_ids):
    result, status = get_ancestors(cid)
    if result is None:
        skipped[status].append(cid)
    else:
        ancestor_sets[cid] = result
    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{len(concept_ids)} processed")
    time.sleep(0.05)

print(f"Done. {len(ancestor_sets)} with ancestors, {len(skipped[404])} skipped (404), {len(skipped[500])} skipped (500).")

# Filter concepts to only those with ancestor data
mask = [cid in ancestor_sets for cid in concept_ids]
concept_ids     = [cid for cid, ok in zip(concept_ids, mask) if ok]
preferred_terms = [t   for t,   ok in zip(preferred_terms, mask) if ok]
fsns            = [f   for f,   ok in zip(fsns, mask) if ok]
df_concepts     = df_concepts[mask].reset_index(drop=True)
print(f"Proceeding with {len(concept_ids)} concepts.")

df_concepts["ancestor_count"] = [len(ancestor_sets[cid]) for cid in concept_ids]
df_concepts.to_csv("data/concepts.csv", index=False)
print("Saved data/concepts.csv")

In [ ]:
def ontological_distance(a_id, b_id):
    anc_a = ancestor_sets[a_id]
    anc_b = ancestor_sets[b_id]
    lca_depth = len(anc_a & anc_b)
    return len(anc_a) + len(anc_b) - 2 * lca_depth

n = len(concept_ids)
print(f"Building {n}×{n} distance matrix...")

# Diagnostic: check ancestor set sizes
anc_sizes = [len(ancestor_sets[cid]) for cid in concept_ids]
print(f"Ancestor set sizes — min: {min(anc_sizes)}, max: {max(anc_sizes)}, mean: {np.mean(anc_sizes):.1f}")
empty_anc = [cid for cid in concept_ids if len(ancestor_sets[cid]) == 0]
if empty_anc:
    print(f"WARNING: {len(empty_anc)} concepts have empty ancestor sets: {empty_anc[:5]}")

D_snomed = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        d = ontological_distance(concept_ids[i], concept_ids[j])
        D_snomed[i, j] = d
        D_snomed[j, i] = d

print(f"Distance matrix shape: {D_snomed.shape}")
pos_vals = D_snomed[D_snomed > 0]
if pos_vals.size > 0:
    print(f"Min: {pos_vals.min():.1f}, Max: {D_snomed.max():.1f}, Mean: {pos_vals.mean():.2f}")
else:
    print("WARNING: all distances are 0 — ancestor sets may be empty or only 1 concept remains")

In [ ]:
pd.DataFrame(D_snomed).to_csv("data/ontological_distances.csv", index=False, header=False)
print("Saved data/ontological_distances.csv")